# Working with OpenINTEL data in pandas/pyarrow

Note that pandas is sufficient for smaller computation loads. OpenINTEL data can be large, and pandas might not be suitable.
Other tools that we have available at DACS include a Spark based compute cluster, and clickhouseDB.

In [3]:
import pandas as pd
import requests
import io

# You also need to have the package 'pyarrow' installed!

In [4]:
# OpenINTEL open data URL, full directory available at: https://openintel.nl/download/, https://openintel.nl/data/forward-dns/top-lists/
openintel_url = 'https://object.openintel.nl/openintel-public/fdns/basis=toplist/source=alexa/year=2022/month=06/day=01/part-00000-56d2956d-0d31-4410-b9cb-722ba814d432-c000.gz.parquet'

In [5]:
# Download the file into memory
response = requests.get(openintel_url)
response.raise_for_status()  # Raise an error if the download fails

In [ ]:
# Read the parquet file into a DataFrame
df = pd.read_parquet(io.BytesIO(response.content), engine="pyarrow")

In [5]:
# Process the OpenINTEL data
alexa_a_df = df[df['response_type'] == 'A'] # Filter on DNS response type A
alexa_a_ip4_df = alexa_a_df['ip4_address']  # Get the column we want

alexa_a_ip4_nparray = alexa_a_ip4_df.unique()  # Only grab unique values

alexa_a_df['domain'] = alexa_a_df['response_name'].str.rstrip('.')

alexa_domain_a_df = alexa_a_df[['domain', 'ip4_address']].copy()
unique_alexa_domain_a_df = alexa_domain_a_df.drop_duplicates()
 # Sometimes duplications are found.
# Print count
print('Total IP4 addresses found: %s and unique addresses are %s' %((len(alexa_a_ip4_df), len(alexa_a_ip4_nparray))))

/tmp/ipykernel_17846/2601080606.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  alexa_a_df['domain'] = alexa_a_df['response_name'].str.rstrip('.')


Total IP4 addresses found: 3239068 and unique addresses are 860648


In [6]:
unique_alexa_domain_a_df.to_csv("../data/alexa-1m/openintel_alexa_domain_a_record_"+day+"_"+mon+"_"+year+".csv", index=False)

In [7]:
# Save ranks of alexa domains in a csv file: alexa_domains_with_rank.csv
# NOTE: This part is not needed for Alexa rank as our data already contains the ranks of the domains.
# import csv

# # Load domains and assign ranks
# rank_dict = {}
# with open("../data/alexa-1m/top-1m-"+year+".text", "r", encoding="utf-8") as f, open("../data/alexa-1m/alexa_domains_with_rank_2022.csv", "w") as out:
#     writer = csv.writer(out)
#     writer.writerow(["domain", "rank"])
    
#     for rank, domain in enumerate(f, start=1):  # Auto-assign rank
#         domain = domain.strip()
#         rank_dict[domain] = str(rank)
#         writer.writerow([domain, rank])

In [8]:
# Find ranks of alexa 1M domains with their A records (IP address/es) and save it into a file.
import csv
import pandas as pd

# 1. Read alexa_domain_a_df that has domain name and ranks: openintel_domain_a_record
# 2. Join it with alexa_domains_with_rank
df1 = pd.read_csv("../data/alexa-1m/alexa_domains_with_rank_"+day+"_"+mon+"_"+year+".csv")

# If the rank file doesnot have headers, insert it.
df1 = pd.read_csv("../data/alexa-1m/alexa_domains_with_rank_"+day+"_"+mon+"_"+year+".csv", header=None, names=['rank', 'domain'])

df2 = pd.read_csv("../data/alexa-1m/openintel_alexa_domain_a_record_"+day+"_"+mon+"_"+year+".csv")
df3 = df1.merge(df2, how='left', on='domain')

df3.to_csv("../data/alexa-1m/openintel_alexa_resolved_with_rank_"+day+"_"+mon+"_"+year+".csv", index=False)

In [9]:
# Find the number of matched IP addresses 

# Compare the ip addresses with the protected prefixes using pytricia loop
import pandas as pd
import ipaddress
import pytricia


def build_prefix_trees(prefix_list):
    """Builds two Patricia Tries: one for IPv4 and one for IPv6."""
    pt_v4 = pytricia.PyTricia()
    pt_v6 = pytricia.PyTricia()

    for prefix in prefix_list:
        network = ipaddress.ip_network(prefix)
        if network.version == 4:

            pt_v4[prefix] = True
        else:
            pt_v6[prefix] = True

    return pt_v4, pt_v6


def count_covered_ips(ip_list, pt_v4, pt_v6):
    """Counts how many IPs are covered by the prefixes in the Patricia Tries."""
    count = 0
    for ip in ip_list:
        ip_obj = ipaddress.ip_address(ip)
        if ip_obj.version == 4 and pt_v4.get(ip):
            count += 1
        elif ip_obj.version == 6 and pt_v6.get(ip):
            count += 1

    return count


def get_covered_ips(ip_list, pt_v4, pt_v6):
    """Returns the list of IPs covered by prefixes in the Patricia Trie."""
    covered_ips = []
    for ip in ip_list:
        ip_obj = ipaddress.ip_address(ip)
        if ip_obj.version == 4 and pt_v4.get(ip):
            covered_ips.append(ip)
        elif ip_obj.version == 6 and pt_v6.get(ip):
            covered_ips.append(ip)
#     covered_ips = [ip for ip in ip_list if prefix_tree.get(ip)]
    return covered_ips



# Protected prefixes lists using all five scrubberscovered_count
df = pd.read_csv("../data/customers_prefixes_scrubber_all_"+year+".csv") 
protected_prefixes = df["prefix"].tolist()

# Openintel list of IP addresses
df = pd.read_csv("../data/alexa-1m/openintel_alexa_resolved_with_rank_"+year+".csv") 
df_cleaned = df.dropna()

alexa_ip_addresses = df_cleaned["ip4_address"].unique()

# Build separate prefix trees for IPv4 and IPv6
pt_v4, pt_v6 = build_prefix_trees(protected_prefixes)

# Find the number of IPs covered
covered_count = count_covered_ips(alexa_ip_addresses, pt_v4, pt_v6)

# Find the IPs covered
covered_ips = get_covered_ips(alexa_ip_addresses, pt_v4, pt_v6)


# Save covered ips in a .txt file.
with open("../data/alexa-1m/openintel_alexa_scrubber_covered_ip_"+year+".txt", "w", encoding="utf-8") as f:
    for ip in covered_ips:
        f.write(f"{ip}\n")
print(f"Number of IPs protected by the five scrubber's protected prefixes is: {len(covered_ips)} and IPs saved in openintel_alexa_scrubber_covered_ip_"+year+".txt")

Number of IPs protected by the five scrubber's protected prefixes is: 3032 and IPs saved in openintel_alexa_scrubber_covered_ip_2017.txt


In [10]:
# Find ranks of tranco 1M domains with their A records (IP address/es) and save it into a file.
import csv

covered_ips = []
# Parse massdns results
with open("../data/alexa-1m/openintel_alexa_scrubber_covered_ip_"+year+".txt", "r", encoding="utf-8") as f:
    for line in f:
        ip = line.strip("\n")
        covered_ips.append(ip)
        
# Convert it into a dataframe with column name ipv4_address
df1 = pd.DataFrame(covered_ips, columns=['ip4_address'])

df2 = pd.read_csv("../data/alexa-1m/openintel_alexa_resolved_with_rank_"+year+".csv")

# df3 = df2.merge(df1, how='inner', on='ip4_address')
        
    
result = df1.merge(df2, on='ip4_address', how='inner')
result.to_csv("../data/alexa-1m/openintel_alexa_ip_domains_ranks_"+year+".csv", index=False)

print("%s number of domains are protected. \n" %len(result))


10776 number of domains are protected. 



In [3]:
import pandas as pd
result = pd.read_csv("../data/after_tma/openintel_alexa_ip_domains_ranks_100k_01_jan_2022.csv")

In [5]:
result.sort_values("rank", ascending=True)

,ip4_address,rank,domain
0,159.45.2.143,147,wellsfargo.com
1,159.45.66.143,147,wellsfargo.com
2,159.45.170.143,147,wellsfargo.com
3,204.135.8.50,175,fedex.com
4,204.135.8.155,175,fedex.com
...,...,...,...
499,198.49.23.144,99928,snorkelmolokini.com
999,198.185.159.144,99961,steakhouse316.com
748,198.49.23.145,99961,steakhouse316.com
500,198.49.23.144,99961,steakhouse316.com
